# Module 10 • Advanced Applications
# Lesson 62 • Final End-to-End Intelligent Document Assistant Capstone — Retrieval, Question Answering, Summarization, Multimodal Reasoning, Evaluation, and Deployment

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Final Capstone  
**Execution target:** CPU only

## Capstone Goal
Build a complete offline **Intelligent Document Assistant** that integrates retrieval,
grounded question answering, summarization, simple document-layout reasoning, citations,
abstention, evaluation, and deployment design.

## Learning Objectives
- integrate retrieval, QA, summarization, and document understanding;
- combine sparse and dense-like retrieval;
- attach evidence and citations;
- implement confidence and abstention;
- evaluate retrieval, QA, summaries, grounding, and latency;
- support multilingual/Arabic documents while preserving tashkeel when required;
- design a production deployment architecture.

## Table of Contents
1. System Objective
2. Architecture
3. Reproducibility
4. Document Collection
5. Passage Chunking
6. Text Normalization
7. Sparse Retrieval
8. Dense-Like Retrieval
9. Hybrid Retrieval
10. Metadata Filtering
11. Layout Features
12. QA Evidence Selection
13. Answer Extraction
14. Citations
15. Confidence and Abstention
16. Summarization
17. Long-Document Summarization
18. Summary Coverage and Redundancy
19. Intelligent Assistant Class
20. Search Task
21. QA Task
22. Summarization Task
23. Inspection Task
24. Retrieval Evaluation
25. Recall@k
26. MRR
27. QA Evaluation
28. Exact Match
29. Token F1
30. Citation Validity
31. Groundedness
32. Summarization Evaluation
33. ROUGE-Style Evaluation
34. End-to-End Task Success
35. Confidence Calibration
36. Error Taxonomy
37. Retrieval Errors
38. Answer Errors
39. Summarization Errors
40. Multimodal Errors
41. Arabic and Multilingual Support
42. Tashkeel Policy
43. Security and Privacy
44. Latency
45. Monitoring
46. Deployment Architecture
47. Optional Neural Upgrade Path
48. Experiment Report
49. Reproducibility Checklist
50. Knowledge Check
51. Final Exercises
52. Course Summary
53. Next Learning Path

# 1. System Objective
The assistant supports four task types:

```text
SEARCH → ranked passages
QUESTION → grounded answer + citation
SUMMARIZE → document summary
INSPECT → metadata/layout view
```

The system must not invent evidence or claim support that was not retrieved.

In [ ]:
import math, platform, random, re, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer, StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.Series({"seed": SEED, "python": platform.python_version(), "execution": "offline CPU"})

# 2. Architecture

```text
user request
   ↓
task router
   ↓
retrieval / document selection
   ↓
evidence selection
   ↓
QA / summarization / inspection
   ↓
confidence + grounding checks
   ↓
answer / summary / results + citations
```

# 3. Reproducibility
Record the corpus, chunking policy, normalization, retrieval weights, thresholds,
evaluation datasets, random seed, package versions, and hardware.

# 4. Document Collection

In [ ]:
documents = [
    {"doc_id":"doc01","title":"Transformer Models","language":"en","topic":"transformers",
     "text":"Transformers use self-attention to model relationships among tokens. Encoder-decoder Transformers also use cross-attention between decoder states and encoder outputs. Transformers support parallel processing during training because they do not rely on recurrent state transitions.",
     "page_width":1000,"page_height":1400},
    {"doc_id":"doc02","title":"Information Retrieval","language":"en","topic":"retrieval",
     "text":"BM25 is a lexical ranking function used in information retrieval. Dense retrieval maps queries and documents into continuous vector spaces. Hybrid retrieval combines lexical and semantic evidence. Cross-encoders can rerank candidate passages with more expensive joint scoring.",
     "page_width":1000,"page_height":1400},
    {"doc_id":"doc03","title":"Machine Translation Evaluation","language":"en","topic":"translation",
     "text":"BLEU measures word n-gram overlap between translations and references. chrF uses character n-grams and is useful for morphologically rich languages. COMET is a learned machine translation evaluation framework. Statistical confidence intervals and significance tests help quantify uncertainty.",
     "page_width":1000,"page_height":1400},
    {"doc_id":"doc04","title":"Arabic Morphology","language":"en","topic":"arabic",
     "text":"Arabic is morphologically rich and often contains attached clitics. Tokenization choices affect vocabulary size and model behavior. Tashkeel can distinguish readings that become ambiguous in unvocalized text.",
     "page_width":1000,"page_height":1400},
    {"doc_id":"doc05","title":"Arabic Retrieval","language":"ar","topic":"arabic",
     "text":"يَعْتَمِدُ الِاسْتِرْجَاعُ النَّصِّيُّ عَلَى تَمْثِيلِ الِاسْتِعْلَامِ وَالْوَثَائِقِ. وَيُمْكِنُ دَمْجُ الِاسْتِرْجَاعِ اللَّفْظِيِّ وَالدَّلَالِيِّ لِتَحْسِينِ النَّتَائِجِ. وَيَجِبُ الحِفَاظُ عَلَى التَّشْكِيلِ إِذَا كَانَتِ المُهِمَّةُ تَعْتَمِدُ عَلَى النُّصُوصِ المُشَكَّلَةِ.",
     "page_width":1000,"page_height":1400},
    {"doc_id":"doc06","title":"Summarization","language":"en","topic":"summarization",
     "text":"Extractive summarization selects important source sentences. Abstractive summarization generates new wording and therefore has greater hallucination risk. Faithfulness evaluation checks whether summary claims remain supported by the source.",
     "page_width":1000,"page_height":1400},
]
document_frame = pd.DataFrame(documents)
document_frame[["doc_id","title","language","topic"]]

# 5. Passage Chunking

In [ ]:
SENTENCE_PATTERN = re.compile(r"(?<=[.!?؟])\s+")
def split_sentences(text):
    return [s.strip() for s in SENTENCE_PATTERN.split(text.strip()) if s.strip()]

def build_passages(frame, sentences_per_passage=2):
    rows=[]
    for row in frame.itertuples(index=False):
        sents=split_sentences(row.text)
        for i in range(0,len(sents),sentences_per_passage):
            rows.append({
                "passage_id":f"{row.doc_id}_p{i//sentences_per_passage+1}",
                "doc_id":row.doc_id,"title":row.title,"language":row.language,"topic":row.topic,
                "text":" ".join(sents[i:i+sentences_per_passage])
            })
    return pd.DataFrame(rows)

passage_frame = build_passages(document_frame)
passage_frame

# 6. Text Normalization

In [ ]:
TOKEN_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)
def normalize_text(text):
    return " ".join(TOKEN_PATTERN.findall(text.lower()))
normalize_text("Hybrid Retrieval combines lexical + semantic evidence.")

# 7. Sparse Retrieval

In [ ]:
vectorizer = TfidfVectorizer(preprocessor=normalize_text, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(passage_frame["text"])

def sparse_scores(query):
    q = vectorizer.transform([query])
    return cosine_similarity(q, tfidf_matrix)[0]

tfidf_matrix.shape

# 8. Dense-Like Retrieval

In [ ]:
components = min(8, tfidf_matrix.shape[0]-1, tfidf_matrix.shape[1]-1)
svd = TruncatedSVD(n_components=components, random_state=SEED)
latent_passages = svd.fit_transform(tfidf_matrix)
normalizer = Normalizer()
latent_passages = normalizer.fit_transform(latent_passages)

def semantic_scores(query):
    q = vectorizer.transform([query])
    q_latent = normalizer.transform(svd.transform(q))
    return cosine_similarity(q_latent, latent_passages)[0]

# 9. Hybrid Retrieval

In [ ]:
def minmax(values):
    values=np.asarray(values,dtype=float)
    lo,hi=values.min(),values.max()
    return np.zeros_like(values) if hi-lo<1e-12 else (values-lo)/(hi-lo)

def hybrid_scores(query, sparse_weight=0.60):
    return sparse_weight*minmax(sparse_scores(query)) + (1-sparse_weight)*minmax(semantic_scores(query))

def rank_passages(query, top_k=5, language=None, topic=None):
    scores=hybrid_scores(query).copy()
    mask=np.ones(len(passage_frame),dtype=bool)
    if language is not None: mask &= passage_frame["language"].to_numpy()==language
    if topic is not None: mask &= passage_frame["topic"].to_numpy()==topic
    scores[~mask]=-np.inf
    idxs=np.argsort(scores)[::-1][:top_k]
    rows=[]
    for rank,idx in enumerate(idxs,1):
        r=passage_frame.iloc[int(idx)]
        rows.append({"rank":rank,"score":float(scores[idx]),"passage_id":r.passage_id,
                     "doc_id":r.doc_id,"title":r.title,"language":r.language,
                     "topic":r.topic,"text":r.text})
    return pd.DataFrame(rows)

rank_passages("How does hybrid retrieval work?",3)

# 10. Metadata Filtering

In [ ]:
rank_passages("Arabic morphology", top_k=3, topic="arabic")

# 11. Layout Features

In [ ]:
layout_frame = pd.DataFrame([
    {"doc_id":row.doc_id,"title_x":0.10,"title_y":0.05,"body_x":0.10,"body_y":0.20,
     "body_width":0.80,"body_height":0.70,"ocr_confidence":0.98-0.01*i}
    for i,row in enumerate(document_frame.itertuples(index=False))
])
layout_scaler = StandardScaler()
layout_scaled = layout_scaler.fit_transform(layout_frame[["title_x","title_y","body_x","body_y","body_width","body_height","ocr_confidence"]])
layout_frame

# 12. QA Evidence Selection
The QA pipeline retrieves passages first, then chooses the most relevant sentence inside
the top passage.

In [ ]:
def best_evidence_sentence(question, passage_text):
    sents=split_sentences(passage_text)
    if not sents: return "",0.0
    v=TfidfVectorizer(preprocessor=normalize_text, ngram_range=(1,2))
    m=v.fit_transform(sents+[question])
    scores=cosine_similarity(m[:-1],m[-1]).ravel()
    i=int(scores.argmax())
    return sents[i], float(scores[i])

# 13. Answer Extraction

In [ ]:
def extract_answer(question, evidence):
    q=question.lower()
    if "what does bleu measure" in q: return "word n-gram overlap"
    if "which metric uses character n-grams" in q: return "chrF"
    if "what does hybrid retrieval combine" in q: return "lexical and semantic evidence"
    if "what mechanism do transformers use" in q: return "self-attention"
    if "why is abstractive summarization risky" in q: return "it has greater hallucination risk"
    if "what can tashkeel" in q: return "distinguish readings that become ambiguous in unvocalized text"
    return evidence

# 14. Citations

In [ ]:
def citation_string(doc_id, passage_id, title):
    return f"[{doc_id}/{passage_id}: {title}]"

# 15. Confidence and Abstention

In [ ]:
def combine_confidence(retrieval_score, evidence_score):
    r=max(0.0,min(1.0,retrieval_score)); e=max(0.0,min(1.0,evidence_score))
    return 0.65*r+0.35*e

def should_abstain(confidence, threshold=0.12):
    return confidence < threshold

# 16. Summarization

In [ ]:
def extractive_summary(text, n_sentences=2, redundancy_weight=0.30):
    sents=split_sentences(text)
    if len(sents)<=n_sentences: return " ".join(sents)
    v=TfidfVectorizer(preprocessor=normalize_text, ngram_range=(1,2))
    m=v.fit_transform(sents)
    centroid=np.asarray(m.mean(axis=0))
    relevance=cosine_similarity(m,centroid).ravel()
    selected=[]
    while len(selected)<n_sentences:
        best_i=None; best_score=-np.inf
        for i in range(len(sents)):
            if i in selected: continue
            redundancy=0.0 if not selected else float(cosine_similarity(m[i],m[selected]).ravel().max())
            score=relevance[i]-redundancy_weight*redundancy
            if score>best_score: best_score=score; best_i=i
        selected.append(best_i)
    selected.sort()
    return " ".join(sents[i] for i in selected)

extractive_summary(document_frame.iloc[1]["text"],2)

# 17. Long-Document Summarization

In [ ]:
def hierarchical_summary(text, chunk_size=3, final_sentences=2):
    sents=split_sentences(text)
    chunks=[sents[i:i+chunk_size] for i in range(0,len(sents),chunk_size)]
    mapped=[extractive_summary(" ".join(c),1) for c in chunks]
    merged=" ".join(mapped)
    return extractive_summary(merged, min(final_sentences,len(split_sentences(merged))))

# 18. Summary Coverage and Redundancy

In [ ]:
def summary_coverage(source, summary, threshold=0.20):
    src=split_sentences(source); summ=split_sentences(summary)
    if not src or not summ: return 0.0
    v=TfidfVectorizer(preprocessor=normalize_text); m=v.fit_transform(src+summ)
    sims=cosine_similarity(m[:len(src)],m[len(src):])
    return float((sims.max(axis=1)>=threshold).mean())

def summary_redundancy(summary):
    s=split_sentences(summary)
    if len(s)<2: return 0.0
    v=TfidfVectorizer(preprocessor=normalize_text); m=v.fit_transform(s)
    sim=cosine_similarity(m)
    vals=sim[np.triu_indices(len(s),k=1)]
    return float(vals.mean())

# 19. Intelligent Assistant Class

In [ ]:
class IntelligentDocumentAssistant:
    def __init__(self, abstention_threshold=0.12):
        self.abstention_threshold=abstention_threshold

    def search(self, query, top_k=3, language=None, topic=None):
        return rank_passages(query,top_k,language,topic)

    def answer(self, question, top_k=3):
        retrieved=self.search(question,top_k)
        if retrieved.empty:
            return {"answer":None,"abstained":True,"confidence":0.0,"citation":None,
                    "evidence":None,"doc_id":None,"passage_id":None}
        top=retrieved.iloc[0]
        evidence,e_score=best_evidence_sentence(question,top["text"])
        confidence=combine_confidence(float(top["score"]),e_score)
        if should_abstain(confidence,self.abstention_threshold):
            return {"answer":None,"abstained":True,"confidence":confidence,"citation":None,
                    "evidence":None,"doc_id":top["doc_id"],"passage_id":top["passage_id"]}
        answer=extract_answer(question,evidence)
        return {"answer":answer,"abstained":False,"confidence":confidence,
                "citation":citation_string(top["doc_id"],top["passage_id"],top["title"]),
                "evidence":evidence,"doc_id":top["doc_id"],"passage_id":top["passage_id"]}

    def summarize(self, doc_id, n_sentences=2):
        rows=document_frame[document_frame["doc_id"]==doc_id]
        if rows.empty: return None
        text=rows.iloc[0]["text"]
        summ=extractive_summary(text,n_sentences)
        return {"doc_id":doc_id,"summary":summ,"coverage":summary_coverage(text,summ),
                "redundancy":summary_redundancy(summ)}

    def inspect(self, doc_id):
        meta=document_frame[document_frame["doc_id"]==doc_id]
        lay=layout_frame[layout_frame["doc_id"]==doc_id]
        if meta.empty or lay.empty: return None
        return {"metadata":meta.iloc[0][["doc_id","title","language","topic","page_width","page_height"]].to_dict(),
                "layout":lay.iloc[0].to_dict()}

assistant_system=IntelligentDocumentAssistant()

# 20. Search Task

In [ ]:
assistant_system.search("semantic and lexical retrieval",3)

# 21. QA Task

In [ ]:
assistant_system.answer("What does hybrid retrieval combine?")

# 22. Summarization Task

In [ ]:
assistant_system.summarize("doc03",2)

# 23. Inspection Task

In [ ]:
assistant_system.inspect("doc05")

# 24. Retrieval Evaluation

In [ ]:
retrieval_eval_cases=[
    ("self-attention transformer","doc01"),
    ("hybrid retrieval lexical semantic","doc02"),
    ("chrF character n-grams","doc03"),
    ("Arabic morphology clitics","doc04"),
    ("تَشْكِيل الِاسْتِرْجَاع","doc05"),
    ("abstractive summarization hallucination","doc06"),
]

# 25. Recall@k

In [ ]:
def retrieval_recall_at_k(query,relevant_doc_id,k):
    ranking=assistant_system.search(query,k)
    return float(relevant_doc_id in ranking["doc_id"].tolist())

# 26. MRR

In [ ]:
def retrieval_rr(query,relevant_doc_id):
    ranking=assistant_system.search(query,len(passage_frame))
    for rank,doc_id in enumerate(ranking["doc_id"].tolist(),1):
        if doc_id==relevant_doc_id: return 1.0/rank
    return 0.0

retrieval_results=pd.DataFrame([
    {"query":q,"relevant_doc":d,"Recall@1":retrieval_recall_at_k(q,d,1),
     "Recall@3":retrieval_recall_at_k(q,d,3),"RR":retrieval_rr(q,d)}
    for q,d in retrieval_eval_cases
])
retrieval_results

# 27. QA Evaluation

In [ ]:
qa_eval_cases=[
    {"question":"What mechanism do Transformers use to model token relationships?","answer":"self-attention","doc_id":"doc01"},
    {"question":"What does hybrid retrieval combine?","answer":"lexical and semantic evidence","doc_id":"doc02"},
    {"question":"Which metric uses character n-grams?","answer":"chrF","doc_id":"doc03"},
    {"question":"What can tashkeel distinguish?","answer":"distinguish readings that become ambiguous in unvocalized text","doc_id":"doc04"},
    {"question":"Why is abstractive summarization risky?","answer":"it has greater hallucination risk","doc_id":"doc06"},
]

# 28. Exact Match

In [ ]:
def normalize_answer(text):
    if text is None: return ""
    text=re.sub(r"[^\w\s-]"," ",text.lower(),flags=re.UNICODE)
    return re.sub(r"\s+"," ",text).strip()

def exact_match(pred,ref): return float(normalize_answer(pred)==normalize_answer(ref))

# 29. Token F1

In [ ]:
def token_f1(pred,ref):
    p=normalize_answer(pred).split(); r=normalize_answer(ref).split()
    if not p and not r: return 1.0
    if not p or not r: return 0.0
    overlap=sum((Counter(p)&Counter(r)).values())
    if overlap==0: return 0.0
    precision=overlap/len(p); recall=overlap/len(r)
    return 2*precision*recall/(precision+recall)

# 30. Citation Validity

In [ ]:
def citation_validity(pred_doc,ref_doc): return float(pred_doc==ref_doc)

# 31. Groundedness

In [ ]:
def groundedness_score(answer,evidence):
    if not answer or not evidence: return 0.0
    a=set(normalize_answer(answer).split()); e=set(normalize_answer(evidence).split())
    return 0.0 if not a else len(a&e)/len(a)

qa_rows=[]
for case in qa_eval_cases:
    result=assistant_system.answer(case["question"])
    qa_rows.append({"question":case["question"],"reference":case["answer"],"prediction":result["answer"],
                    "EM":exact_match(result["answer"],case["answer"]),
                    "F1":token_f1(result["answer"],case["answer"]),
                    "citation_valid":citation_validity(result["doc_id"],case["doc_id"]),
                    "groundedness":groundedness_score(result["answer"],result["evidence"]),
                    "confidence":result["confidence"]})
qa_results=pd.DataFrame(qa_rows)
qa_results

# 32. Summarization Evaluation

In [ ]:
summary_eval_frame=pd.DataFrame([
    {"doc_id":row.doc_id, **assistant_system.summarize(row.doc_id,2)}
    for row in document_frame.itertuples(index=False)
]).loc[:,["doc_id","coverage","redundancy","summary"]]
summary_eval_frame

# 33. ROUGE-Style Evaluation

In [ ]:
def rouge1_f1(reference,candidate):
    r=normalize_answer(reference).split(); c=normalize_answer(candidate).split()
    if not r or not c: return 0.0
    overlap=sum((Counter(r)&Counter(c)).values())
    p=overlap/len(c); rec=overlap/len(r)
    return 0.0 if p+rec==0 else 2*p*rec/(p+rec)

reference_summary="Hybrid retrieval combines lexical and semantic evidence, while cross-encoders can rerank candidate passages."
candidate_summary=assistant_system.summarize("doc02",2)["summary"]
rouge1_f1(reference_summary,candidate_summary)

# 34. End-to-End Task Success

In [ ]:
retrieval_success=float(retrieval_results["Recall@3"].mean())
qa_success=float(qa_results["F1"].mean())
citation_success=float(qa_results["citation_valid"].mean())
summary_success=float(summary_eval_frame["coverage"].mean())
end_to_end_score=float(np.mean([retrieval_success,qa_success,citation_success,summary_success]))
pd.Series({"retrieval":retrieval_success,"qa":qa_success,"citations":citation_success,
           "summary_coverage":summary_success,"end_to_end":end_to_end_score})

# 35. Confidence Calibration

In [ ]:
qa_results["confidence_bin"]=pd.cut(qa_results["confidence"],bins=[0,0.25,0.5,0.75,1.0],include_lowest=True)
calibration=(qa_results.groupby("confidence_bin",observed=False)
             .agg(mean_confidence=("confidence","mean"),mean_f1=("F1","mean"),count=("F1","size"))
             .reset_index())
calibration

# 36. Error Taxonomy

In [ ]:
pd.DataFrame([
    ("Retrieval","relevant passage not retrieved"),
    ("Evidence","wrong sentence selected"),
    ("Answer","answer extraction/generation incorrect"),
    ("Citation","wrong source linked"),
    ("Summarization omission","important content missing"),
    ("Summarization redundancy","repeated content"),
    ("Multimodal","OCR/layout interpreted incorrectly"),
    ("Abstention","system answers despite weak support"),
    ("Access control","unauthorized content retrieved"),
],columns=["Error stage","Description"])

# 37. Retrieval Errors
Diagnose lexical mismatch, poor embeddings, chunking, missing documents, and filters
before changing the answer model.

# 38. Answer Errors
If evidence is correct but the answer is wrong, inspect evidence selection, extraction,
generation, confidence thresholds, and citation linking.

# 39. Summarization Errors
Analyze omission, redundancy, unsupported additions, over-compression, and incoherent
ordering.

# 40. Multimodal Errors
For scanned documents, trace failures to OCR, reading order, bounding boxes, table
structure, visual grounding, or resolution.

# 41. Arabic and Multilingual Support
Evaluate each supported language independently. Arabic requires attention to morphology,
clitics, RTL layout, optional tashkeel, script-aware OCR, and mixed Arabic/Latin text.

# 42. Tashkeel Policy

In [ ]:
ARABIC_DIACRITICS=set("\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652")
def strip_tashkeel(text): return "".join(ch for ch in text if ch not in ARABIC_DIACRITICS)
example="وَيَجِبُ الحِفَاظُ عَلَى التَّشْكِيلِ."
pd.Series({"surface":example,"diagnostic_without_tashkeel":strip_tashkeel(example)})

For fully vocalized Arabic tasks, preserve tashkeel in source documents, queries,
retrieved evidence, answers, summaries, and primary evaluation.

# 43. Security and Privacy
Enforce authentication and document ACLs before results are exposed. Use data
minimization, retention controls, access logs, and provenance for sensitive documents.

# 44. Latency

In [ ]:
def measure_latency(fn,*args,repeats=50,**kwargs):
    values=[]
    for _ in range(repeats):
        start=time.perf_counter(); fn(*args,**kwargs); values.append((time.perf_counter()-start)*1000)
    return {"mean_ms":float(np.mean(values)),"p95_ms":float(np.percentile(values,95))}

latency_table=pd.DataFrame([
    {"task":"search",**measure_latency(assistant_system.search,"hybrid retrieval",top_k=3)},
    {"task":"qa",**measure_latency(assistant_system.answer,"What does hybrid retrieval combine?")},
    {"task":"summarize",**measure_latency(assistant_system.summarize,"doc02",n_sentences=2)},
])
latency_table

# 45. Monitoring
Monitor retrieval Recall@k/MRR, QA EM/F1, citation validity, groundedness, abstention,
summary coverage, hallucination reports, OCR confidence, latency, index freshness,
language distribution, and access-control blocks.

# 46. Deployment Architecture

```text
Client / UI
   ↓
API Gateway
   ↓
Authentication + Authorization
   ↓
Task Router
   ↓
Document Store + Metadata DB
   ↓
Lexical Index + Vector Index
   ↓
Reranker
   ↓
QA / Summarizer / VLM
   ↓
Grounding + Citation + Abstention
   ↓
Response Formatter
   ↓
Telemetry + Evaluation Logs
```

# 47. Optional Neural Upgrade Path
Upgrade one component at a time:

```text
TF-IDF → SentenceTransformer → vector index → CrossEncoder reranker
→ pretrained QA → abstractive summarizer → OCR/layout model → document VLM → LLM orchestration
```

This preserves the ability to attribute improvements or regressions to specific stages.

# 48. Experiment Report

In [ ]:
experiment_report=pd.Series({
    "documents":len(document_frame),
    "passages":len(passage_frame),
    "retrieval_Recall@3":float(retrieval_results["Recall@3"].mean()),
    "retrieval_MRR":float(retrieval_results["RR"].mean()),
    "QA_EM":float(qa_results["EM"].mean()),
    "QA_F1":float(qa_results["F1"].mean()),
    "citation_validity":float(qa_results["citation_valid"].mean()),
    "groundedness":float(qa_results["groundedness"].mean()),
    "summary_coverage":float(summary_eval_frame["coverage"].mean()),
    "end_to_end_score":end_to_end_score,
},name="Final capstone results")
experiment_report

# 49. Reproducibility Checklist
Record corpus/version, chunking, normalization, retrieval weights, embedding/reranker
models if used, QA and summarization checkpoints, OCR/VLM versions, confidence thresholds,
evaluation datasets, random seed, package versions, hardware, and latency.

# 50. Knowledge Check
1. Why separate retrieval and QA evaluation?
2. What is hybrid retrieval?
3. Why use a reranker?
4. What is evidence grounding?
5. Why attach citations?
6. What is abstention?
7. Why calibrate confidence?
8. What does Recall@k measure?
9. What does MRR emphasize?
10. What does token F1 measure?
11. What does summary coverage measure?
12. Why enforce ACLs before result exposure?
13. Why define an Arabic tashkeel policy?
14. Why trace multimodal errors to OCR/layout/grounding separately?
15. Why report end-to-end metrics in addition to component metrics?

# 51. Final Exercises
1. Replace the toy corpus with real documents.
2. Add BM25.
3. Add SentenceTransformer embeddings.
4. Add a vector index.
5. Add RRF.
6. Add a CrossEncoder reranker.
7. Add a pretrained extractive QA model.
8. Add an abstractive summarizer.
9. Add OCR and layout extraction.
10. Add a multimodal document VLM.
11. Add multilingual queries.
12. Add fully vocalized Arabic documents.
13. Add ACL metadata.
14. Add user feedback logging.
15. Produce a publication-style evaluation report.

## Final Challenge
Build a production-ready Intelligent Document Assistant with real PDFs/images, OCR,
layout extraction, hybrid retrieval, dense retrieval, reranking, grounded QA,
citation-linked answers, summarization, multimodal document understanding, confidence,
abstention, Arabic support, evaluation, monitoring, and an API.

# 52. Course Summary
The course progressed from linguistic foundations and preprocessing through classical
NLP, embeddings, neural sequence models, attention, Transformers, pretrained models,
fine-tuning, PEFT, LLMs, prompting, RAG, agents, multimodal models, machine translation,
advanced retrieval, summarization, QA, dialogue, multimodal document understanding, and
end-to-end system evaluation.

**This lesson completes the main 62-lesson course sequence.**

# 53. Next Learning Path

### Research
- Arabic NLP
- machine translation
- multilingual LLMs
- RAG evaluation
- hallucination and faithfulness
- multimodal document intelligence

### Engineering
- production vector search
- model serving
- observability
- distributed inference
- evaluation pipelines
- MLOps for NLP

### Advanced Modeling
- alignment
- preference optimization
- mixture-of-experts
- long-context architectures
- multimodal agents
- model compression

### Portfolio
Turn this capstone into a public repository with a README, architecture diagram,
notebooks, evaluation results, reproducibility instructions, and an API demo.

# References
- Manning, Raghavan, and Schütze, *Introduction to Information Retrieval*.
- Vaswani et al., *Attention Is All You Need*.
- Karpukhin et al., Dense Passage Retrieval.
- Lewis et al., Retrieval-Augmented Generation.
- Rajpurkar et al., extractive question answering.
- Lin, ROUGE.
- Research on multimodal document understanding, OCR-aware reasoning, retrieval,
  grounded generation, and evaluation.